# LangChain Memory 시리즈 7/7: VectorStoreRetrieverMemory

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- 임베딩(Embedding)과 벡터 유사도 검색의 원리를 설명할 수 있어요
- Chroma 벡터 스토어를 초기화하고 `VectorStoreRetrieverMemory`를 구성할 수 있어요
- `k` 값으로 검색 결과 개수를 조절할 수 있어요
- 순서 기반 메모리와 의미 기반 메모리의 차이를 비교하고 선택 기준을 세울 수 있어요
- Memory 시리즈 7종 전체를 비교해 서비스 요건에 맞는 메모리를 선택할 수 있어요

## 사전 지식

| 개념 | 설명 |
|------|------|
| 임베딩 (Embedding) | 텍스트를 의미를 담은 숫자 벡터로 변환하는 기법 |
| 벡터 유사도 (Vector Similarity) | 두 벡터 간 코사인 유사도 등을 계산해 의미적 가까움을 측정 |
| Chroma | 오픈소스 벡터 데이터베이스. 인메모리 또는 디스크 저장 모두 지원 |
| RAG (Retrieval-Augmented Generation) | 검색으로 찾은 문서를 LLM 입력에 추가하는 패턴 (6_RAG 섹션 참고) |

## 순서 기반 vs 의미 기반 메모리

1~6번 노트북의 메모리는 모두 **대화 순서**를 기준으로 저장·삭제해요. 최신 대화가 중요하고 수백 턴이 넘는 장기 대화에서는 오래된 정보를 아예 잃어버려요.

`VectorStoreRetrieverMemory`는 순서와 관계없이 **의미적 유사성**으로 검색해요. 100턴 전에 나눈 대화라도 현재 질문과 의미가 가깝다면 바로 꺼내올 수 있어요.

> 🔑 **핵심 개념**: VectorStoreRetrieverMemory는 1~6번 메모리와 근본적으로 다릅니다. 순서(시간)가 아닌 **의미적 관련성**으로 기억을 꺼내는 방식이며, RAG(검색 증강 생성) 패턴의 메모리 버전입니다.

> 💡 **실무 팁**: 고객 서비스 봇처럼 수천 번의 대화가 누적되는 장기 애플리케이션에 적합합니다. 단, 임베딩 API 비용과 벡터 DB 비용이 추가로 발생합니다.

```mermaid
flowchart LR
    subgraph Sequential["순서 기반 메모리 (1~6번)"]
        S1[1턴] --> S2[2턴] --> S3[...] --> S100[100턴]
        S100 --> LATEST[최근 N턴만 사용]
    end

    subgraph Semantic["의미 기반 메모리 (이번 노트북)"]
        V1[1턴 벡터]
        V2[2턴 벡터]
        V100[100턴 벡터]
        Q[현재 질문] --"코사인 유사도"--> V2
        Q --"코사인 유사도"--> V100
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    class S1,S2,S3 output
    class S100,LATEST output
    class V1,V2,V100 process
    class Q input
```

In [1]:
# ---------------------------------------------------
# 환경 변수 로드 및 벡터 스토어 메모리 관련 모듈 임포트
# ---------------------------------------------------
from dotenv import load_dotenv
from langchain.memory import VectorStoreRetrieverMemory
# OpenAIEmbeddings: 텍스트를 벡터로 변환하는 임베딩 모델
from langchain_openai import OpenAIEmbeddings
# Chroma: 오픈소스 벡터 데이터베이스 (인메모리 또는 디스크 저장 지원)
from langchain_community.vectorstores import Chroma

load_dotenv()

True

## 1. 기본 사용법 - 벡터 스토어 메모리 생성

먼저 벡터 스토어를 초기화하고, `VectorStoreRetrieverMemory`를 생성합니다.


### 💡 VectorStoreRetrieverMemory 초기화 흐름

```mermaid
graph TD
    LOAD_ENV --> EMBED["OpenAIEmbeddings()"]
    EMBED --> VS["Chroma(embedding_function=embeddings)"]
    VS --> RETR["vectorstore.as_retriever(k=1)"]
    RETR --> MEM_INIT["VectorStoreRetrieverMemory(retriever)"]
```

In [2]:
# ---------------------------------------------------
# 임베딩 모델 → Chroma 벡터 스토어 → Retriever → 메모리 순서로 초기화
# ---------------------------------------------------

# ============================================================
# TODO: OpenAIEmbeddings, Chroma, Retriever, VectorStoreRetrieverMemory를 순서대로 생성하세요
# 힌트:
#   1. embeddings = OpenAIEmbeddings()
#   2. vectorstore = Chroma(embedding_function=embeddings)  # 빈 인메모리 스토어
#   3. retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
#   4. memory = VectorStoreRetrieverMemory(retriever=retriever)
# 예상 결과: memory.save_context() 호출 시 대화가 벡터로 변환되어 Chroma에 저장됨
# ============================================================

# 1단계: 임베딩 모델 생성
# - OpenAIEmbeddings: 텍스트를 고차원 벡터로 변환 (의미적 유사성 계산에 사용)
embedings = OpenAIEmbeddings()

# 2단계: 벡터 스토어 초기화 (빈 인메모리 스토어)
# - Chroma(embedding_function=embeddings): 빈 상태로 인메모리 벡터 스토어 생성
# - persist_directory를 지정하지 않으면 인메모리로 동작 (커널 종료 시 데이터 소멸)
verctorstore = Chroma(embedding_function=embedings)

# 3단계: Retriever 생성 (상위 k개 문서 검색)
# - k=1: 가장 유사한 대화 1개만 반환 (토큰 절약)
retriever = verctorstore.as_retriever(search_kwargs={"k": 1})

# 4단계: VectorStoreRetrieverMemory 생성
# - retriever를 기반으로 save_context()/load_memory_variables() 구현
memory = VectorStoreRetrieverMemory(retriever=retriever)

/var/folders/k2/q6_f_rp50lqg7l24d_3dk1vh0000gn/T/ipykernel_24196/4114910956.py:22: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  verctorstore = Chroma(embedding_function=embedings)
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
/var/folders/k2/q6_f_rp50lqg7l24d_3dk1vh0000gn/T/ipykernel_24196/4114910956.py:30: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = VectorStoreRetrieverMemory(retriever=retriever)


### 💡 벡터 스토어에 대화 저장

```mermaid
flowchart TD
    subgraph SaveLoop
        H[사용자 발화] --> SAVE[save_context]
        A[AI 응답] --> SAVE
    end

    SAVE --> EMBED_SAVE[임베딩 변환]
    EMBED_SAVE --> VS_UPSERT[Chroma 벡터 스토어에 저장]
```

### 대화 저장 및 의미 기반 검색

대화를 저장하고, 질문에 따라 의미적으로 관련된 메모리를 검색합니다.


### 💡 의미 기반 검색 루프 (k=1)

```mermaid
flowchart LR
    QUESTIONS[[질문 리스트]] --> LOOP[for question in questions]
    LOOP --> EMBED_Q[질문 임베딩]
    EMBED_Q --> RETRIEVE[retriever.get_relevant_documents]
    RETRIEVE --> MEMORY_LOAD["load_memory_variables({'human': question})"]
    MEMORY_LOAD --> PRINT_RESULT[관련 history 출력]
```



In [3]:
# ---------------------------------------------------
# 여러 대화를 벡터 스토어에 저장
# ---------------------------------------------------
# save_context() 호출 시 대화가 임베딩 벡터로 변환되어 Chroma에 추가됨
# 순서가 아닌 의미적 유사도 기반으로 나중에 검색됨
conversations = [
    {
        "human": "안녕하세요! 오늘 프로젝트 회의에 참석해주셔서 감사합니다. 자기소개 부탁드립니다.",
        "ai": "안녕하세요. 저는 데이터 과학을 전공한 시니어 데이터 분석가입니다. 주로 파이썬과 SQL을 사용하며, 머신러닝 모델 개발과 데이터 시각화에 전문성을 가지고 있습니다."
    },
    {
        "human": "프로젝트에서 어떤 역할을 맡으실 계획인가요?",
        "ai": "제가 맡을 역할은 데이터 파이프라인 구축과 분석 모델 개발입니다. 고객 데이터를 수집하고 처리하여 인사이트를 도출하는 작업을 담당할 예정입니다. 또한 팀원들과 협력하여 분석 결과를 시각화하고 보고서를 작성하겠습니다."
    },
    {
        "human": "팀 프로젝트에서 어려움을 겪었던 경험이 있다면 어떻게 해결하셨나요?",
        "ai": "이전 프로젝트에서 데이터 품질 문제로 어려움을 겪었습니다. 이를 해결하기 위해 데이터 검증 프로세스를 도입하고, 팀원들과 정기적인 데이터 리뷰 미팅을 가졌습니다. 또한 자동화된 데이터 품질 체크 도구를 개발하여 문제를 사전에 발견할 수 있도록 했습니다."
    },
    {
        "human": "데이터 분석가로서 자신의 강점은 무엇이라고 생각하시나요?",
        "ai": "제 강점은 복잡한 데이터를 이해하기 쉬운 인사이트로 변환하는 능력입니다. 통계적 분석과 머신러닝 기법을 활용하여 숨겨진 패턴을 발견하고, 이를 비즈니스 의사결정에 도움이 되는 정보로 제공할 수 있습니다. 또한 팀워크를 중시하며 다양한 이해관계자와 효과적으로 소통합니다."
    }
]

# 모든 대화 저장 (각 대화가 임베딩 벡터로 변환되어 Chroma에 추가됨)
for conv in conversations:
    memory.save_context(
        inputs={"human": conv["human"]},
        outputs={"ai": conv["ai"]},
    )



### 💡 k 값 변경 후 검색 흐름 (k=2)

```mermaid
flowchart TD
    RETR_INIT["기존 retriever(k=1)"] -->|search_kwargs={'k':2}| RETR_K2[새 retriever]
    RETR_K2 --> MEM_K2[VectorStoreRetrieverMemory]
    MEM_K2 --> SAVE_K2[동일 대화 저장]
    SAVE_K2 --> RETRIEVE_K2["load_memory_variables({'human': 질문})"]
    RETRIEVE_K2 --> OUTPUT_K2[상위 2개 문서 history]
```

### 의미 기반 메모리 검색

질문에 따라 의미적으로 가장 관련성 높은 메모리를 검색합니다.


### 💡 순서 기반 vs 의미 기반 비교 흐름

```mermaid
flowchart TD
    DATASEQ[[test_conversations]] --> BUF_SAVE[buffer_memory.save_context]
    DATASEQ --> VEC_SAVE[vector_memory.save_context]

    BUF_SAVE --> BUF_HISTORY[history = 전체 메시지]
    VEC_SAVE --> VEC_INDEX[벡터 스토어 인덱스]

    BUF_HISTORY --> BUF_LOAD["load_memory_variables({})"]
    VEC_INDEX --> VEC_LOAD["load_memory_variables({'human': '두 번째'})"]

    BUF_LOAD --> REPORT_BUF[모든 메시지 수 출력]
    VEC_LOAD --> REPORT_VEC[질문과 의미적으로 가까운 문서만 반환]
```



### 한눈에 비교: BufferMemory vs VectorStoreRetrieverMemory

100턴 대화 중 1턴에서 나눈 **"파이썬 전문"** 정보를 나중에 다시 찾으려면 어떻게 해야 할까요?

| 항목 | BufferMemory (순서 기반) | VectorStoreRetrieverMemory (의미 기반) |
|------|--------------------------|----------------------------------------|
| **검색 방법** | 전체 100턴을 프롬프트에 넣어야 함 | "파이썬 관련 대화" 질문 → 의미적으로 가장 가까운 k개만 반환 |
| **토큰 사용량** | 100턴 × 평균 50토큰 = **~5,000토큰** | 1턴 × 평균 50토큰 = **~50토큰** (k=1 기준) |
| **오래된 정보** | WindowMemory 사용 시 잘려나감 (유실) | 100턴 전 대화도 의미가 맞으면 즉시 검색 |
| **비용** | 메모리 저장 비용 없음 | 임베딩 API 호출 비용 + 벡터 DB 비용 발생 |

> **핵심 포인트**: 대화가 길어질수록 BufferMemory는 토큰 비용이 선형으로 증가하지만, VectorStoreRetrieverMemory는 항상 k개만 반환하므로 **토큰 비용이 일정**합니다. 장기 대화 애플리케이션에서 특히 유리해요.

In [4]:
# ---------------------------------------------------
# 의미 기반 메모리 검색 (k=1)
# ---------------------------------------------------
# load_memory_variables({"human": 질문}): 질문을 임베딩으로 변환 후 코사인 유사도로 검색
# 저장 순서와 관계없이 의미적으로 가장 가까운 대화를 반환
questions = [
    "데이터 분석가의 전공은 무엇인가요?",
    "프로젝트에서 맡을 역할은 무엇인가요?",
    "어려움을 어떻게 해결했나요?",
    "강점은 무엇인가요?"
]


memory_vars = memory.load_memory_variables({"human": questions[2]})
memory_vars

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{'history': 'human: 팀 프로젝트에서 어려움을 겪었던 경험이 있다면 어떻게 해결하셨나요?\nai: 이전 프로젝트에서 데이터 품질 문제로 어려움을 겪었습니다. 이를 해결하기 위해 데이터 검증 프로세스를 도입하고, 팀원들과 정기적인 데이터 리뷰 미팅을 가졌습니다. 또한 자동화된 데이터 품질 체크 도구를 개발하여 문제를 사전에 발견할 수 있도록 했습니다.'}

## 2. k 값 조정하기

`k` 값을 조정하여 더 많은 관련 메모리를 검색할 수 있습니다.


## 3. 다른 메모리 타입과의 차이

`VectorStoreRetrieverMemory`는 대화 순서를 추적하지 않고 의미 기반으로 검색합니다.


In [ ]:
# ---------------------------------------------------
# ConversationBufferMemory(순서 기반) vs VectorStoreRetrieverMemory(의미 기반) 비교
# ---------------------------------------------------
# - BufferMemory: 대화를 순서대로 저장, load 시 전체 반환
# - VectorStoreRetrieverMemory: 벡터로 저장, load 시 질문과 유사한 k개만 반환

from langchain.memory import ConversationBufferMemory

# 비교용 대화
test_conversations = [
    {"human": "첫 번째 대화입니다.", "ai": "첫 번째 답변입니다."},
    {"human": "두 번째 대화입니다.", "ai": "두 번째 답변입니다."},
    {"human": "세 번째 대화입니다.", "ai": "세 번째 답변입니다."},
]

# ConversationBufferMemory: 순서대로 저장 (제한 없음)


# VectorStoreRetrieverMemory: 의미 기반 검색 (k=1)


# 비교


## 4. 실전 예제: VectorStoreRetrieverMemory + LLM 챗봇

지금까지는 `save_context()`와 `load_memory_variables()`를 수동으로 호출했어요. 이제 **실제 LLM과 연결**해서 대화하는 챗봇을 만들어 봅시다.

벡터 메모리 챗봇의 동작 원리:
1. 사용자 질문이 들어오면, 벡터 스토어에서 **의미적으로 가장 관련 있는 과거 대화**를 검색
2. 검색된 대화를 프롬프트의 `{history}` 자리에 넣어서 LLM에 전달
3. LLM이 과거 맥락을 참고하여 답변 생성
4. 새 대화를 다시 벡터 스토어에 저장

> 이 방식은 수백, 수천 턴의 대화에서도 **현재 질문과 관련된 맥락만** 프롬프트에 포함하므로 토큰을 효율적으로 사용합니다.

In [7]:
# ---------------------------------------------------
# VectorStoreRetrieverMemory + LLM 챗봇 구성
# ---------------------------------------------------
# 기존 vectorstore와 embeddings를 재사용하여 챗봇을 만듭니다.
# ConversationChain 대신 직접 프롬프트 + 메모리를 조합합니다.

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# ============================================================
# TODO: 벡터 메모리 챗봇을 구성하세요
# 힌트:
#   1. Chroma + Retriever(k=2) + VectorStoreRetrieverMemory 생성
#   2. ChatOpenAI(model="gpt-4o-mini", temperature=0) 초기화
#   3. ChatPromptTemplate에 {history}와 {input} 변수 포함
#   4. RunnablePassthrough.assign()으로 history를 메모리에서 로드하는 체인 구성
# 예상 결과: chatbot_chain.invoke({"input": "질문"})으로 대화가 가능해야 합니다
# ============================================================

# 1단계: 새로운 벡터 메모리용 Chroma + Retriever 생성 (깨끗한 상태에서 시작)
chatbot_vectorstore = Chroma(embedding_function=embedings)
chatbot_retriever = chatbot_vectorstore.as_retriever(search_kwargs={"k": 2})
chatbot_memory = VectorStoreRetrieverMemory(retriever=retriever)

# 2단계: LLM 모델 초기화
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 3단계: 프롬프트 템플릿 생성
# - {history}: 벡터 메모리에서 검색된 관련 과거 대화가 들어가는 자리
# - {input}: 현재 사용자 질문
chatbot_prompt = ChatPromptTemplate.from_template(
  """
  당신은 친절한 AI 비서입니다. 아래 관련 대화 기록을 참고하여 질문에 답변하세오.
  **관련 대화 기록이 없으면 일반 지식으로 답변하세요.**

  관련 대화기록:
  {history}

  현재 질문:
  {input}

  답변:
  """
)

# 4단계: 체인 구성 (메모리 로드 → 프롬프트 → LLM → 출력 파서)
chatbot_chain = (
  RunnablePassthrough.assign(
    history=RunnableLambda(
      lambda x : chatbot_memory.load_memory_variables({"human": x["input"]}).get("history", "")
    )
  )
  | chatbot_prompt
  | llm
  | StrOutputParser()
)

print("벡터 메모리 챗봇이 준비되었습니다!")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


벡터 메모리 챗봇이 준비되었습니다!


In [10]:
# ---------------------------------------------------
# 벡터 메모리 챗봇으로 3번 대화하기
# ---------------------------------------------------
# 대화 주제: 파이썬 웹 프레임워크 → 다른 주제 → 다시 파이썬 관련 질문
# 마지막 질문에서 의미 기반 검색으로 첫 번째 대화를 찾아오는지 확인합니다.

# ============================================================
# TODO: chat() 헬퍼 함수를 만들고, 3번의 대화를 실행하세요
# 힌트:
#   1. chat(user_input) 함수: chatbot_chain.invoke() + chatbot_memory.save_context()
#   2. 1번째 대화: 파이썬 웹 프레임워크 질문
#   3. 2번째 대화: 전혀 다른 주제 (예: 요리)
#   4. 3번째 대화: 다시 파이썬 관련 질문 → 1번째 대화가 검색되는지 확인
# 예상 결과: 3번째 대화에서 AI가 1번째 대화의 맥락을 참고하여 답변
# ============================================================

# 여기에 코드를 작성하세요
def chat(user_input):
    """챗봇에 질문하고 결과를 메모리에 저장하는 함수"""
    response=chatbot_chain.invoke({"input": user_input})
    chatbot_memory.save_context(
        inputs={"human":user_input},
        outputs={"ai": response}
    )

    return response

q1 = "블록체인에서 이더리움과 비트코인의 차이점을 알려줘."
a1 = chat(q1)
print(f' ==> [Line 30]: \033[38;2;53;162;73m[a1]\033[0m({type(a1).__name__}) = \033[38;2;248;247;166m{a1}\033[0m')

q2 = "너가 방금 대답한 내용을 요약해봐"
a2 = chat(q2)
print(f' ==> [Line 33]: \033[38;2;44;253;23m[a2]\033[0m({type(a2).__name__}) = \033[38;2;178;45;17m{a2}\033[0m')

q3 = "엥 무슨 말을 하는거야?"
a3 = chat(q3)
print(f' ==> [Line 37]: \033[38;2;151;126;165m[a3]\033[0m({type(a3).__name__}) = \033[38;2;32;0;183m{a3}\033[0m')


 ==> [Line 30]: [a1](str) = 이더리움과 비트코인은 모두 블록체인 기술을 기반으로 한 암호화폐이지만, 그 목적과 기능에서 몇 가지 중요한 차이점이 있습니다.

1. **목적**:
   - **비트코인**: 주로 디지털 통화로서의 역할을 수행하며, 가치 저장 및 교환 수단으로 설계되었습니다. 비트코인은 탈중앙화된 디지털 화폐로서의 기능에 중점을 둡니다.
   - **이더리움**: 스마트 계약과 분산 애플리케이션(DApps)을 실행할 수 있는 플랫폼을 제공하는 것을 목표로 합니다. 이더리움은 프로그래머블 블록체인으로, 다양한 애플리케이션을 구축할 수 있는 환경을 제공합니다.

2. **기술적 차이**:
   - **스마트 계약**: 이더리움은 스마트 계약을 지원하여 자동화된 계약 실행을 가능하게 합니다. 비트코인은 기본적으로 이러한 기능을 제공하지 않습니다.
   - **블록 생성 시간**: 비트코인의 블록 생성 시간은 약 10분인 반면, 이더리움은 약 15초로 더 빠릅니다.

3. **통화**:
   - **비트코인**: 비트코인(BTC)이라는 단일 암호화폐를 사용합니다.
   - **이더리움**: 이더(ETH)라는 암호화폐를 사용하며, 네트워크에서 거래 수수료 및 계산 비용을 지불하는 데 사용됩니다.

4. **합의 알고리즘**:
   - **비트코인**: 작업 증명(Proof of Work, PoW) 알고리즘을 사용합니다.
   - **이더리움**: 초기에는 PoW를 사용했으나, 최근에는 지분 증명(Proof of Stake, PoS)로 전환하여 에너지 효율성을 높이고 네트워크 확장성을 개선하고자 합니다.

이러한 차이점들은 이더리움과 비트코인이 각각의 강점과 용도를 가지게 하며, 다양한 블록체인 기반 솔루션을 가능하게 합니다.
 ==> [Line 33]: [a2](str) = 저는 데이터 과학을 전공한 시니어 데이터 분석가로, 파이썬과 SQL을 사용하여 머신러닝 모델 개발과 데이터 시각화에 전문성을 가지고 있습니다.
 ==> [Line 

In [11]:
# ---------------------------------------------------
# 마지막 질문에서 벡터 메모리가 어떤 과거 대화를 검색했는지 확인
# ---------------------------------------------------
# "파이썬 프레임워크 초보자 추천" 질문에 대해
# 김치찌개 대화가 아닌, 파이썬 프레임워크 대화가 검색되었는지 확인합니다.

# ============================================================
# TODO: 벡터 메모리 검색 결과를 확인하세요
# 힌트: chatbot_memory.load_memory_variables({"human": 질문})으로 검색 결과 확인
# 예상 결과: 파이썬 프레임워크 관련 대화가 검색되어야 합니다 (김치찌개가 아님)
# ============================================================

# 여기에 코드를 작성하세요
retrieved = chatbot_memory.load_memory_variables({"human": q2})
print(f' ==> [Line 14]: \033[38;2;205;191;26m[retrieved]\033[0m({type(retrieved).__name__}) = \033[38;2;239;95;3m{retrieved}\033[0m')


 ==> [Line 14]: [retrieved](dict) = {'history': 'human: 너가 방금 대답한 내용을 요약해봐\nai: 저는 데이터 과학을 전공한 시니어 데이터 분석가로, 파이썬과 SQL을 사용하여 머신러닝 모델 개발과 데이터 시각화에 전문성을 가지고 있습니다.'}


## 핵심 정리

```python
# VectorStoreRetrieverMemory 기본 사용법
from langchain.memory import VectorStoreRetrieverMemory
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 벡터 스토어 초기화
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(embedding_function=embeddings)

# Retriever 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# 메모리 생성
memory = VectorStoreRetrieverMemory(retriever=retriever)

# 대화 저장
memory.save_context(
    inputs={"human": "사용자 메시지"},
    outputs={"ai": "AI 응답"}
)

# 의미 기반 검색
memory_vars = memory.load_memory_variables({"human": "질문"})
```

### 주요 특징
- ✅ **의미 기반 검색**: 벡터 임베딩을 사용하여 의미적으로 관련된 메모리 검색
- ✅ **순서 무관**: 대화 순서를 명시적으로 추적하지 않음
- ✅ **장기 메모리**: 대량의 메모리를 효율적으로 저장하고 검색 가능
- ✅ **관련성 기반**: 질문과 관련성 높은 메모리만 반환

### 다른 메모리 타입과의 차이

| 특징 | ConversationBufferMemory | VectorStoreRetrieverMemory |
|------|-------------------------|---------------------------|
| 저장 방식 | 순서대로 모든 대화 저장 | 벡터 스토어에 저장 |
| 검색 방식 | 순서 기반 | 의미 기반 (벡터 검색) |
| 반환 내용 | 모든 대화 | 관련성 높은 k개만 |
| 대화 순서 | 중요함 | 중요하지 않음 |
| 적합한 경우 | 짧은 대화, 순서 중요 | 긴 대화, 의미적 관련성 중요 |

### 주의사항
- ⚠️ **벡터 스토어 필요**: Chroma, FAISS 등의 벡터 스토어가 필요함
- ⚠️ **임베딩 모델**: 임베딩 모델의 성능에 따라 검색 품질이 달라짐
- ⚠️ **k 값 선택**: 너무 작으면 정보 부족, 너무 크면 노이즈 증가
- ⚠️ **순서 정보 손실**: 대화 순서 정보가 손실될 수 있음